In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import types
import pyspark.sql.functions as F

In [12]:
# Create a spark session
spark = SparkSession.builder.master("local[*]").appName("taxi-rides-app").getOrCreate()

print("[INFO] Starting spark session")

if not spark.version:
    print("[ERROR] Could not start SPARK session. Exiting program.")
    import os

    exit(1)

[INFO] Starting spark session


In [13]:
from common.config import get_root_path

# Declare dataset paths
DATA_PATH = get_root_path() / "data"
TAXI_PATH = DATA_PATH / "taxi"
DATASET_CLEAN_PATH = TAXI_PATH / "clean" / "yellow" / "2025" / "11"
DATASET_REPORT_PATH = TAXI_PATH / "report"

In [14]:
print(f"[INFO] Reading clean dataset yellow/2025/11")
dataset_file_path = str(DATASET_CLEAN_PATH)
df = spark.read.parquet(dataset_file_path)

print(f"[INFO] Reading taxi zone lookup dataset")
lookup_file_path = str(DATA_PATH / "taxi_zone_lookup.csv")
lookup_df = spark.read.csv(lookup_file_path, header=True)

[INFO] Reading clean dataset yellow/2025/11
[INFO] Reading taxi zone lookup dataset


In [15]:
lookup_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [16]:
join_col = df["pickup_location_id"] == lookup_df["LocationID"]
df_joined = df.join(lookup_df, on=join_col)
df_joined.show()

+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+------------+-----------+----------+---------+--------------------+------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|rate_code_id|pickup_location_id|dropoff_location_id|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|service_type|airport_fee|LocationID|  Borough|                Zone|service_zone|
+---------+-------------------+-------------------+------------------+------------+------------------+-------------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+------

In [ ]:
print(f"[INFO] Processing report: homework")

report_df = spark.createDataFrame(
    [
        ("spark-version", spark.version),
        ("parquet-avg-size", df.count())
    ],
    ["name", "value"],
)

# total_rows = F.lit(df.count()).cast(types.IntegerType())

# report_df = (
#     df.withColumn("spark-version", F.lit(spark.version).cast(types.StringType()))
#     .withColumn("parquet-avg-size", total_rows)
#     # .withColumn("nov-15th-trips-count", None)
#     # .withColumn("longest-trip-in-hours", None)
#     # .withColumn("spark-ui-port", F.lit("4040"))
#     # .withColumn("least-frequent-pickup-zone", None)
# )

# report_df = (
#     df.select(
#         "spark-version",
#         "parquet-avg-size",
#     )
# )

report_df.show()

[INFO] Processing report: homework
+----------------+-------+
|            name|  value|
+----------------+-------+
|   spark-version|  4.1.1|
|parquet-avg-size|4181444|
+----------------+-------+



In [ ]:
# print(f"[INFO] Loading report: homework")
# output_path = str(DATASET_REPORT_PATH / "homework")
# df.repartition(4).write.parquet(
#     path=output_path,
#     mode="overwrite",
# )

In [ ]:
# spark.stop()